## 00 - Install Packages

### Aim
* One-time setup of the R/Bioconductor environment needed for the TCGA-PAAD TPM download pipeline (notebooks 01-03), plus a recorded, verified-safe installation recipe for the three TME deconvolution tools planned in notebook 04 (XDec, CIBERSORTx, TIMER).

### Input
* None. Checks the currently installed R packages in this environment.

### Output
* `TCGAbiolinks`, `SummarizedExperiment` (Bioconductor) and `dplyr`, `tidyr`, `readr`, `ggplot2`, `pheatmap` (CRAN) installed into the active R library, if not already present.
* A documented (not auto-executed) install recipe for XDec/CIBERSORTx/TIMER, chosen specifically to avoid disrupting the existing `pixi` global `r-base`/`python` environments - see "Deconvolution tools" section below.

### Steps
* Ensure `BiocManager` is available.
* Install any missing Bioconductor packages (GDC query/download/prepare + SummarizedExperiment container).
* Install any missing CRAN packages used for data wrangling and QC plotting.
* Record the verified install path for each of the three deconvolution tools (XDec, CIBERSORTx, TIMER), each using whichever mechanism keeps it tracked by `pixi`/conda rather than a side-channel installer (`pip`/`remotes::install_github`) that conda doesn't know about.

### Result
* Idempotent: re-running this notebook skips packages that are already installed.
* Deconvolution tools section is informational/commands-only - running those `pixi global install` / `git clone` commands is left to the user, since they modify the shared global `pixi` environment rather than this notebook's own working directory.

### Conclusion
* Environment is ready for notebook 01 (download), 02 (matrix build) and 03 (QC). The deconvolution tools (notebook 04 onward) each have exactly one missing piece - `quadprog` (XDec, conda-forge) and `r-immunedeconv` (TIMER, bioconda) - both installable via `pixi global install` without touching anything already in the environment; CIBERSORTx needs no package install at all (Docker is already present, only a token registration remains).
</cell id="f86dcddb">


### Set working directory to the project root

In [1]:
# Jupyter's R kernel starts with the notebook's own folder as the working
# directory. This pipeline's root is progress/ (scripts/ and data/ live
# nested under progress/, not the repo root) - every path below is
# relative to that.
if (basename(getwd()) == "rnaseq_download") setwd(file.path("..", ".."))
getwd()

[1] "/Users/liruixi/Desktop/i20_transfer/QCB/rotation/Alex/XDec_Pancreatic_Cancer"

### Ensure BiocManager is available

In [2]:
if (!requireNamespace("BiocManager", quietly = TRUE)) {
  install.packages("BiocManager")
}

### Install Bioconductor packages (GDC query/download/prepare)

In [3]:
bioc_pkgs <- c(
  "TCGAbiolinks",        # GDC query/download/prepare
  "SummarizedExperiment" # container returned by GDCprepare
)

to_install_bioc <- bioc_pkgs[!bioc_pkgs %in% rownames(installed.packages())]
if (length(to_install_bioc)) BiocManager::install(to_install_bioc, update = FALSE, ask = FALSE)

### Install CRAN packages (data wrangling + QC plotting)

In [4]:
cran_pkgs <- c(
  "dplyr",
  "tidyr",
  "readr",
  "ggplot2",
  "pheatmap"
)

to_install_cran <- cran_pkgs[!cran_pkgs %in% rownames(installed.packages())]
if (length(to_install_cran)) install.packages(to_install_cran)

### Deconvolution tools: XDec, CIBERSORTx, TIMER - easiest & safest install path

Recorded here for notebook 04 onward. None of the three tools has a conda package under its own name (checked directly against conda-forge/bioconda/dnachun) - but each one's *actual remaining gap* turns out to be small and conda-trackable, so there's no need for `pip`, `remotes::install_github()`, or a separate environment. These are shell commands (not R) - run them in a terminal, not this notebook; they modify the shared global `pixi` environment, not this project's local files.

**XDec (`BRL-BCM/XDec_SM`) - Python. No package for the tool itself; it's a `git clone`.**
```bash
git clone https://github.com/BRL-BCM/XDec_SM.git
```
Dependencies `numpy`/`pandas`/`scipy` are already in the `python` pixi env; the one gap is `quadprog` (confirmed on conda-forge):
```bash
pixi global install quadprog --environment python
```
Run it with that env's interpreter (not exposed globally by default):
```bash
~/.pixi/envs/python/bin/python XDec_SM/python/xdec.py stage1 --bulk ...
```

**TIMER - via `immunedeconv`, R, conda-packaged (do not `git clone hanfeisun/TIMER` or use `remotes::install_github`).**
`hanfeisun/TIMER` is maintainer-archived and depends on an external Dropbox link for reference data - skip it. `remotes::install_github("omnideconv/immunedeconv")` would write outside conda's tracking and risks silently upgrading a pinned dependency. Instead, the prebuilt package already exists on bioconda and resolves cleanly against everything already in `r-base`:
```bash
pixi global install r-immunedeconv --environment r-base
```
Then in R: `immunedeconv::deconvolute(expr_mat, method = "timer", indications = rep("paad", ncol(expr_mat)))`.

**CIBERSORTx - no package of any kind, in any language.**
No public source exists for the actual algorithm (confirmed: zero matches on every channel, and the GitHub repos that mention "CIBERSORTx" are either unrelated wrappers for the older CIBERSORT or I/O helper scripts, not the engine). Docker is already installed on this machine (v27.3.1) - the only remaining step is registering a free token at `https://cibersortx.stanford.edu`, then:
```bash
docker run cibersortx/fractions ...
```

In [ ]:
# Deconvolution tool installs are shell/pixi commands, not R - see the markdown
# cell above for the exact, verified-safe commands. Nothing to run in this cell;
# kept so the notebook's structure still matches "note package, then install cell".

In [6]:
cat("Environment ready.\n")

Environment ready.
